# Build the Direct Lake semantic model (`Fabric_Governance`)

Creates a **Direct Lake** semantic model over the gold governance tables in
`lh_fabric_management` (`fabricmanagement` schema) and adds measures for the
report.

## Prerequisites
- The **gold notebook** has run, so `gold_file_dependencies`, `gold_report_risk`, and `report_model_map` exist.
- The workspace's **XMLA endpoint = Read Write** (needed for the measure step; on by default for Fabric capacities).
- Uses **semantic-link-labs** (installed in the first cell).

In [ ]:
# Install semantic-link-labs, then bump PyJWT back up so it doesn't conflict with
# the base image's fsspec-wrapper (which needs PyJWT>=2.6.0). Clears the pip
# "dependency conflicts" error. The kernel auto-restarts after %pip -- that's
# expected (not an error); just run the cells below afterwards.
%pip install -q semantic-link-labs
%pip install -q "PyJWT>=2.6.0"

In [ ]:
# ── Create the Direct Lake model over the gold tables ────────────────────────
from sempy_labs.directlake import generate_direct_lake_semantic_model

MODEL  = "Fabric_Governance"
LH     = "lh_fabric_management"
SCHEMA = "fabricmanagement"
# {model table name : schema-qualified lakehouse table}  (schema-enabled lakehouse)
TABLES = {
    "gold_file_dependencies": f"{SCHEMA}.gold_file_dependencies",
    "gold_report_risk":       f"{SCHEMA}.gold_report_risk",
    "report_model_map":       f"{SCHEMA}.report_model_map",
}

generate_direct_lake_semantic_model(
    dataset=MODEL,
    tables=TABLES,
    source=LH,
    source_type="Lakehouse",
    overwrite=True,
    refresh=True,
)
print(f"Direct Lake model '{MODEL}' created over {list(TABLES)}")

In [ ]:
# ── Add measures (TOM) ───────────────────────────────────────────────────────
from sempy_labs.tom import connect_semantic_model

MONEY = "#,0"

with connect_semantic_model(dataset=MODEL, readonly=False) as tom:
    # report-risk measures
    tom.add_measure("gold_report_risk", "Reports",
                    "DISTINCTCOUNT(gold_report_risk[report_id])", format_string=MONEY)
    tom.add_measure("gold_report_risk", "High-Risk Reports",
                    'CALCULATE(DISTINCTCOUNT(gold_report_risk[report_id]), FILTER(gold_report_risk, gold_report_risk[high_sources] > 0))',
                    format_string=MONEY)
    tom.add_measure("gold_report_risk", "Total Risk Score",
                    "SUM(gold_report_risk[risk_score])", format_string=MONEY)
    tom.add_measure("gold_report_risk", "Avg Risk Score",
                    "AVERAGE(gold_report_risk[risk_score])", format_string="#,0.0")
    # file-dependency measures
    tom.add_measure("gold_file_dependencies", "Files",
                    "DISTINCTCOUNT(gold_file_dependencies[path])", format_string=MONEY)
    tom.add_measure("gold_file_dependencies", "High-Risk Files",
                    'CALCULATE(DISTINCTCOUNT(gold_file_dependencies[path]), FILTER(gold_file_dependencies, gold_file_dependencies[risk_tier] = "High"))',
                    format_string=MONEY)
    tom.add_measure("gold_file_dependencies", "Max Models per File",
                    "MAX(gold_file_dependencies[models_using])", format_string=MONEY)

print("Measures added.")

## Verify — model tables + measures

In [ ]:
from sempy_labs.tom import connect_semantic_model
print("Measures on Fabric_Governance:")
with connect_semantic_model(dataset="Fabric_Governance", readonly=True) as tom:
    for m in tom.all_measures():
        print(f"  [{m.Parent.Name}]  {m.Name}")

## Build the report (in the Fabric UI)

Generating the report from hand-authored report JSON (`create_report_from_reportjson`)
proved too version-sensitive — it created the item but the visuals wouldn't render
("loading your report" hang). The reliable path is to build it on the model in the UI
(~2 min, and it looks better):

1. Open the **`Fabric_Governance`** semantic model → **Auto-create report** (one-click
   starter) *or* **New report** (blank).
2. **Table visual — files by dependents** (from `gold_file_dependencies`):
   `path`, `source_kind`, `source_host`, `risk_tier`, `models_using`, `reports_using`
   — sort by `models_using` descending.
3. **Table visual — riskiest reports** (from `gold_report_risk`):
   `report_name`, `workspace_name`, `high_sources`, `medium_sources`, `risk_score`
   — sort by `risk_score` descending, filter **Top 10** by `risk_score`.
4. (Optional) Add cards using the measures: `High-Risk Reports`, `Total Risk Score`,
   `High-Risk Files`, `Max Models per File`.
5. **Save as `Fabric Governance Report`.**